In [ ]:
from datetime import datetime
import pandas as pd
from matplotlib import pyplot as plt
import pycountry
from os import listdir as ls

from emu_renewal.inputs import OUTPUTS_PATH, DATA_PATH, get_google_mobility
from emu_renewal.plotting import compare_proc_versus_mobility, MOB_COLOURS
from emu_renewal.utils import get_countries_by_continent

In [ ]:
job_path = OUTPUTS_PATH / "44255911"
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=[15, 15])
flat_axes = axes.ravel()
for c, country in enumerate(countries_by_cont["NA"]):
    c_ax = flat_axes[c]
    country_name = pycountry.countries.lookup(country).name
    c_ax.set_title(country_name)
    proc_samples = pd.read_hdf(job_path / country / "no_mob/spaghetti.h5")["process"]
    centiles = proc_samples.quantile([0.05, 0.5, 0.95], axis=1).T
    c_ax.plot(centiles.index, centiles[0.5], label="process", color="navy")
    c_ax.fill_between(centiles.index, centiles[0.05], centiles[0.95], alpha=0.2, color="navy")
    mob = get_google_mobility(country)
    mobility = mob.loc[mob.index < centiles.index[-1]]
    smoothed_mob = mob.rolling(7, center=True).mean().dropna()
    c_ax.plot(smoothed_mob.index, smoothed_mob["retail_and_recreation"], color=MOB_COLOURS["g_mob"])
    plt.setp(c_ax.xaxis.get_majorticklabels(), rotation=70)
fig.tight_layout()